<a href="https://colab.research.google.com/github/samaeldomany/-MultiModal-Document-Intelligence-RAG-Based-QA-System-/blob/main/MultiMedia_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dependancies

In [2]:
!apt-get install -y poppler-utils
!pip install transformers==4.46.1 peft==0.13.2 qdrant-client colpali_engine torch torchvision Pillow gradio google-genai pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [10]:
import os
from pdf2image import convert_from_path
import glob
import torch
import uuid
from PIL import Image
from colpali_engine.models import ColPaliProcessor, ColPali
from qdrant_client import QdrantClient, models
import gradio as gr
from google import genai
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# PDF Preprocessing

In [11]:

pdf_folder_path = "/content/drive/MyDrive/Multimedia assignment/"
output_dir = "document_pages"
os.makedirs(output_dir, exist_ok=True)
metadata_list = []

pdf_files = glob.glob(os.path.join(pdf_folder_path, "*.pdf"))

print(f"Found {len(pdf_files)} PDF files in {pdf_folder_path}")

for pdf_path in pdf_files:
    print(f"Extracting pages from {os.path.basename(pdf_path)}...")

    pages = convert_from_path(pdf_path, dpi=200)
    doc_name = os.path.basename(pdf_path)

    for i, page in enumerate(pages):
        page_num = i + 1
        image_path = os.path.join(output_dir, f"{doc_name}_page_{page_num}.jpg")
        page.save(image_path, 'JPEG')

        metadata_list.append({
            "file_path": image_path,
            "document_name": doc_name,
            "page_number": page_num
        })

print(f"Successfully saved {len(metadata_list)} images total from all PDFs.")

Found 3 PDF files in /content/drive/MyDrive/Multimedia assignment/
Extracting pages from The Federal Reserve Financial Stability Report (April 2024).pdf...
Extracting pages from Central Bank of Egypt (CBE) Financial Stability Report (March 2025).pdf...
Extracting pages from IMF Global Financial Stability Report (April 2024).pdf...
Successfully saved 305 images total from all PDFs.


# Vector Indexing with ColPali

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware utilized: {device}")

model_name = "vidore/colpali-v1.3"
processor = ColPaliProcessor.from_pretrained(model_name)
model = ColPali.from_pretrained(model_name, torch_dtype=torch.bfloat16).eval().to(device)

client = QdrantClient(":memory:")
collection_name = "financial_reports"

client.create_collection(
    collection_name,
    vectors_config={
        "colpali": models.VectorParams(
            size=model.dim,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM
            )
        )
    }
)


print("Starting vector indexing...")
for meta in metadata_list:
    image = Image.open(meta["file_path"])

    batch_images = processor.process_images([image]).to(device)
    with torch.no_grad():
        image_embedding = model(**batch_images)[0].to(torch.float32).cpu().numpy()

    client.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=uuid.uuid4().hex,
                vector={"colpali": image_embedding},
                payload=meta
            )
        ]
    )
print("Indexing complete! Qdrant database is ready.")

Hardware utilized: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Starting vector indexing...
Indexing complete! Qdrant database is ready.


# RAG Pipeline and Gradio UI

In [14]:
ai_client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

def answer_question(query):
    batch_queries = processor.process_queries([query]).to(device)
    with torch.no_grad():
        query_embedding = model(**batch_queries)[0].to(torch.float32).cpu().numpy()

    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        using="colpali",
        limit=1,
        with_payload=True
    ).points

    if not results:
        return "No documents found in the database.", None

    payload = results[0].payload
    image_path = payload["file_path"]
    retrieved_image = Image.open(image_path)

    prompt = f"""
    You are an expert financial analyst. Analyze the provided document page.
    Answer the following question accurately based STRICTLY on the charts, tables, or text in the image.
    If the answer is not in the image, say "I cannot answer based on the provided document."

    Question: {query}
    """

    response = ai_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[prompt, retrieved_image]
    )

    citation = f"\n\n---\n**Source:** `{payload['document_name']}`, Page {payload['page_number']}"
    return response.text + citation, image_path


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("#  MultiModal Document Intelligence Chatbot")
    gr.Markdown("Ask questions about the financial report. The system retrieves the exact page using ColPali and analyzes it using Gemini.")

    with gr.Row():
        with gr.Column():
            query_input = gr.Textbox(label="Enter your question:")
            submit_btn = gr.Button("Search & Generate", variant="primary")
            answer_output = gr.Markdown(label="Generated Answer")

        with gr.Column():
            image_output = gr.Image(label="Retrieved Evidence", type="filepath")

    submit_btn.click(fn=answer_question, inputs=query_input, outputs=[answer_output, image_output])

demo.launch(debug=True)

/tmp/ipykernel_24396/1808758962.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b9db0a399c45dd013e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b9db0a399c45dd013e.gradio.live
